In [2]:
import os
os.chdir('/home/jiangwenbin/shared/workspace/AlphaCore/examples/voicebank')
print(os.getcwd())

/home/jiangwenbin/shared/workspace/AlphaCore/examples/voicebank


In [3]:
import shutil
from pathlib import Path
import librosa
import numpy as np
import librosa.display
import matplotlib.pyplot as plt

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import seaborn
seaborn.set_theme(style="whitegrid", font_scale=1.2)
plt.rcParams['axes.grid'] = False

import sys
sys.path.append("../../")
from modules.utils import metrics

sr = 16000
n_fft = 512
win_length=400
hop_length=100

save_dir = Path('exp/paper/MOS-GAN')

In [32]:
idx_list = ['p257_008', 'p257_033', 'p257_106', 'p232_250','p232_409']

wav_list = [("exp/logs/UnSE/3a0e/clean/{}.wav", "Clean"),
            ("exp/logs/UnSE/3a0e/noisy/{}.wav", "Noisy"),
            ("exp/denoise/OMLSA_voicebank_testset/{}.wav", "OMLSA"),
            ("exp/denoise/DCCRN_voicebank/{}.wav", "DCCRN"),
            ("exp/logs/IterSE/3h0/est/{}.wav", "NyTT"),
            ("exp/logs/IterSE/3b21/est/{}.wav", "RemixIT"),
            ("exp/logs/UnSE/3f3d/est/{}.wav", "MOS-GAN"),            
            ("exp/logs/UnSE/3h4/est/{}.wav", r"MOS-GAN_3h4"),
            ("exp/logs/UnSE/3h5/est/{}.wav", r"MOS-GAN_3h5"),
            ("exp/logs/UnSE/3h6/est/{}.wav", r"MOS-GAN_3h6")]
for file_idx in idx_list:
    for (wav, method) in wav_list:
        wav_path = Path(wav.format(file_idx))
        dest_dir = save_dir.joinpath(f"samples/{method}")
        dest_dir.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(wav_path, dest_dir.joinpath(wav_path.name))

In [27]:
_, model_list = zip(*wav_list)
html_str = "<tr><td>{}</td>"
for model in model_list:    
    print("<tr> <td>", model, "</td>")
    for file_idx in idx_list:
        wav_path = Path("samples").joinpath(model, "{}.wav".format(file_idx))
        print("""<td><audio controls=""><source src="{}"></audio></td>""".format(wav_path))
    print("</tr>")

<tr> <td> Clean </td>
<td><audio controls=""><source src="samples/Clean/p257_008.wav"></audio></td>
<td><audio controls=""><source src="samples/Clean/p257_033.wav"></audio></td>
<td><audio controls=""><source src="samples/Clean/p257_106.wav"></audio></td>
<td><audio controls=""><source src="samples/Clean/p232_250.wav"></audio></td>
<td><audio controls=""><source src="samples/Clean/p232_409.wav"></audio></td>
</tr>
<tr> <td> Noisy </td>
<td><audio controls=""><source src="samples/Noisy/p257_008.wav"></audio></td>
<td><audio controls=""><source src="samples/Noisy/p257_033.wav"></audio></td>
<td><audio controls=""><source src="samples/Noisy/p257_106.wav"></audio></td>
<td><audio controls=""><source src="samples/Noisy/p232_250.wav"></audio></td>
<td><audio controls=""><source src="samples/Noisy/p232_409.wav"></audio></td>
</tr>
<tr> <td> OMLSA </td>
<td><audio controls=""><source src="samples/OMLSA/p257_008.wav"></audio></td>
<td><audio controls=""><source src="samples/OMLSA/p257_033.wav">

In [4]:
figsize = (15, 12.5)
nrows, ncols = 5, 2
file_idx = "p232_409"
fig, ax = plt.subplots(nrows=nrows, ncols=ncols, sharex=True, figsize=figsize)
wav_list = [(f"exp/logs/UnSE/3a0e/clean/{file_idx}.wav", "Clean"),
            (f"exp/logs/UnSE/3a0e/noisy/{file_idx}.wav", "Noisy"),
            (f"exp/denoise/OMLSA_voicebank_testset/{file_idx}.wav", "OMLSA"),
            (f"exp/denoise/DCCRN_voicebank/{file_idx}.wav", "DCCRN"),
            (f"exp/logs/IterSE/3h0/est/{file_idx}.wav", "NyTT"),
            (f"exp/logs/IterSE/3b21/est/{file_idx}.wav", "RemixIT"),
            (f"exp/logs/UnSE/3f3d/est/{file_idx}.wav", "MOS-GAN"),            
            (f"exp/logs/UnSE//3h4/est/{file_idx}.wav", r"w/o $L_{spp}$"),
            (f"exp/logs/UnSE//3h5/est/{file_idx}.wav", r"w/o $L_{fid}$"),
            (f"exp/logs/UnSE//3h6/est/{file_idx}.wav", r"w/o $L_{p}$"),]
clean, _ = librosa.load(wav_list[0][0], sr=None)
for idx, (wav, method) in enumerate(wav_list):
    row_idx, col_idx = idx//ncols, idx%ncols
    y, sr = librosa.load(wav, sr=None)
    stft_y = librosa.stft(y, n_fft=n_fft)
    D = librosa.amplitude_to_db(np.abs(stft_y), ref=np.max)
    img = librosa.display.specshow(D, y_axis='linear', x_axis='s', sr=sr, hop_length=hop_length,
                                   ax=ax[row_idx, col_idx], cmap=('jet'))
    if idx == 0:
        title = method
    else:
        ref, est = clean, y
        est = est[:ref.shape[0]]
        ref = ref[:est.shape[0]]
        score = metrics.eval(ref, est, sr)
        title = "{:}  {{PESQ: {:.2f}, eSTOI: {:.2f}, SI_SNR: {:.2f}}}".format(method, score['PESQ'], 
                                                                              score['eSTOI'], score['SI_SNR'])
        print(title)
    ax[row_idx, col_idx].set(title=title)

for ax_i in ax:
    for ax_ij in ax_i:
        ax_ij.label_outer()
fig.colorbar(img, ax=ax, format="%+2.f dB")
fig.subplots_adjust(left=0,bottom=0.1,top=0.9,right=0.77,hspace=0.2,wspace=0.05)
fig.savefig(save_dir.joinpath(f"img/{file_idx}.png"), dpi=120, bbox_inches='tight')

Noisy  {PESQ: 1.27, eSTOI: 0.66, SI_SNR: 1.32}
OMLSA  {PESQ: 1.48, eSTOI: 0.64, SI_SNR: 12.78}
DCCRN  {PESQ: 1.91, eSTOI: 0.81, SI_SNR: 15.84}
NyTT  {PESQ: 1.38, eSTOI: 0.67, SI_SNR: 6.17}
RemixIT  {PESQ: 1.58, eSTOI: 0.70, SI_SNR: 10.00}
MOS-GAN  {PESQ: 1.48, eSTOI: 0.68, SI_SNR: 12.63}
w/o $L_{spp}$  {PESQ: 1.23, eSTOI: 0.64, SI_SNR: 1.31}
w/o $L_{fid}$  {PESQ: 1.48, eSTOI: 0.68, SI_SNR: 12.66}
w/o $L_{p}$  {PESQ: 1.05, eSTOI: 0.06, SI_SNR: -6.95}
